# Complete RAG Pipeline — Architecture

```
---------------- Architecture Diagram ----------------

[Raw Text / Documents]
        |
        v
[OllamaEmbeddings(model="nomic-embed-text")]
        |
        v
[embed_query("What is RAG?")]   [embed_documents(chunks)]
        |                               |
        v                               v
  [Single Vector 768-dim]     [List of Vectors 768-dim each]
        |                               |
        v                               v
[Chroma.from_documents(docs, embedding=embeddings)]
        |
        v
[ChromaDB — In-Memory Vector Store]
        |
        |---------> [similarity_search(query, k=1)]
        |                   |
        |                   v
        |           [Top-k Matching Documents]
        |
        |---------> [as_retriever(search_kwargs={"k": 1})]
                        |
                        v
                [retriever.invoke(query)]
                        |
                        v
                [List[Document] — Retrieved Results]
                        |
                        v
                [doc.page_content → Final Answer]

---------------- Deep Architecture Notes (Telugu-English Mix) ----------------

1. OllamaEmbeddings:
   - Idi locally run avutundi — Ollama server meeru start chesthe, idi
     "nomic-embed-text" model use chesi text ni numbers (vectors) ga convert chestundi.
   - embed_query() → oka single sentence ki oka vector istundi (768 numbers).
   - embed_documents() → anni chunks ki vectors istundi (list of lists).

2. Chroma Vector Store:
   - Chroma oka vector database — idi documents ni vaati embeddingsతో patu store chestundi.
   - from_documents() call chesthe, documents embed avutayi and in-memory lo store avutayi.
   - Disk ki save cheyyakunda RAM lo untundi — fast ga work chestundi.

3. similarity_search():
   - Query ni embed chesi, stored vectors తో compare chestundi.
   - k=1 ante most similar oka document return avutundi.
   - Idi cosine similarity use chestundi — angle chinna ga unte similar.

4. as_retriever():
   - Vectorstore ni LangChain Retriever ga wrap chestundi.
   - invoke() method use chesi query pass chesthe, relevant documents return avutundi.
   - Idi LangChain chains తో directly connect cheyyachu — RAG pipeline lo use avutundi.

5. Final Output:
   - doc.page_content → retrieve chesina document text print avutundi.
   - Idi LLM ki context ga pass cheyyadam next step — complete RAG lo.
```

In [1]:
! ollama --version

ollama version is 0.32.1


In [4]:
! ollama pull nomic-embed-text
%pip install langchain-chroma -q

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector = embeddings.embed_query(
    "What is RAG?"
)

print(type(vector))
print(len(vector))
print(vector[:10])

<class 'list'>
768
[0.05835144, 0.009220345, -0.17474803, -0.12459035, 0.009547923, -0.0144671975, 0.02647738, 0.041882556, 0.0029675693, -0.081537105]


In [7]:
! ollama list

NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    44 seconds ago    
llama3.2:latest            a80c4f17acd5    2.0 GB    4 weeks ago       


In [8]:
chunks = [
    "RAG helps LLMs access external knowledge.",
    "Embeddings convert text into vectors.",
    "Vector databases store embeddings."
]

vectors = embeddings.embed_documents(
    chunks
)

print(len(vectors))

print(len(vectors[0]))

print(vector)

3
768
[0.05835144, 0.009220345, -0.17474803, -0.12459035, 0.009547923, -0.0144671975, 0.02647738, 0.041882556, 0.0029675693, -0.081537105, 0.0008111351, 0.0121486625, 0.047871944, 0.019202057, 0.0056313807, -0.049723204, 0.004677063, -0.03162282, -0.053905644, 0.016452927, -0.02484407, -0.050731152, 0.043965634, 0.0017276627, 0.13083471, -0.019504646, -0.037667252, -0.033107392, 0.022952037, 0.015687402, -0.016715704, -0.061018586, -0.0018383586, -0.042775333, 0.06656153, 0.011045995, 0.052988697, 0.009670819, -0.015348346, 0.012831618, -0.027065983, 0.04536404, 0.03507697, -0.008471288, 0.056743834, 0.043849055, 0.035508007, 0.0254081, 0.023549201, -0.03798916, 0.008565636, -0.080992594, 0.022868013, -0.018497366, 0.06363692, -0.024433397, 0.011210629, -0.013855239, -0.029722825, 0.06399012, 0.033008303, 0.028513843, -0.09953789, 0.07088869, 0.012044784, -0.017739424, -0.0080190655, -0.0022718753, -0.008348077, -0.048479367, 0.055405244, -0.008390371, 0.059787463, 0.033222515, -0.0286

In [6]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

docs = [
    Document(
        page_content="RAG helps LLMs access external knowledge."
    ),
    Document(
        page_content="Embeddings convert text into vectors."
    ),
    Document(
        page_content="Vector databases store embeddings."
    )
]

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings
)


In [7]:
vectorstore

In [8]:
results = vectorstore.similarity_search(
    "How do LLMs access external knowledge?",
    k=1
)

for doc in results:
    print(doc.page_content)

RAG helps LLMs access external knowledge.


In [9]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 1}
)

docs = retriever.invoke(
    "what is the role of vector Database?"
)

for doc in docs:
    print(doc.page_content)

Vector databases store embeddings.
